# Module 4 Jupyter Lab

This Jupyter notebook is a place to practice skills that leverage the OILER framework. Remember to
use the [OILER Framework Book](https://umsi-mads.github.io/framework-book/) as a reference when in
need.

__Warning__: You are likely to encounter errors in this notebook. This is intentional. Please use
these errors as an opportunity to practice your debugging skills.

Note: certain `assert` statements rely on static fixture data provided by the module
`c1m3_fxt_data.py`.

In [ ]:
import csv
import json
import c1m4_fxt_data as fxt
import pprint

# Configure a PrettyPrinter object
pp = pprint.PrettyPrinter(indent=2, sort_dicts=False, width=120)

__Check the docs__: The standard library's `pprint` module provides "pretty print" CLI capabilities.
If you have never used the module, consider reviewing the official Python
[documentation](https://docs.python.org/3/library/pprint.html).

## Skill practice: orienting yourself

Review the Jupyter notebook. Your first task is to orient yourself to both the data and the
notebook's execution flow. Consider the following questions:

* __Data__: Review the data file. What format is employed? Is the data structured or unstructured?
  What entities are represented? What entity attributes are included and how are they typed?

* __Execution flow__: Review the notebook's structure. What is the purpose of each code cell? What
  computations are performed? If the cell features one or more function definitions, review each
  function's name, parameter list, body, and `return` statement (if any). Determine the sequence of
  operations each function performs, including what inputs are required and what outputs are
  produced. If the cell includes one or more function calls, review the name and argument list in
  the expression. Determine what value is returned (if any), its type, and the variable(s) in play
  as a result of the call.

* __Markdown__: Review the accompanying Markdown cells for information and/or instructions.

## Skill practice: data retrieval

This lab explores player shooting performance during the 2023 FIFA Women's World Cup hosted by
Australia and New Zealand. Thirty-two (`32`) teams competed in the tournament with Spain emerging as
the champion.

### Read/Write files

The player data was sourced from [FBREF](https://fbref.com/en/). Before commencing the lab, review
the file named "data-2023-fifa_wwc-players.csv" stored in the `data` subdirectory. Note how the data
is structured. Examine the "headers" row.

Three functions are provided to read from and write to files:

`read_csv()`: Reads a CSV file, parsing row values per the provided delimiter. Returns a list of
lists, wherein each nested list represents a single row from the input file. If a byte order mark
(BOM) is encountered at the beginning of the first line of decoded text, call `read_csv()` and pass
"utf-8-sig" as the `encoding` argument.

`write_csv()`: Writes data to a target CSV file. Column headers are written as the first
row of the CSV file if optional `headers` are specified.

`write_json()`: Serializes data as JSON. Writes content to the provided filepath.

In [ ]:
def read_csv(filepath, encoding="utf-8", newline="", delimiter=","):
    with open(filepath, "r", encoding=encoding, newline=newline) as file_obj:
        return [row for row in csv.reader(file_obj, delimiter=delimiter)]


def write_csv(filepath, data, headers=None, encoding="utf-8", newline=""):
    with open(filepath, "w", encoding=encoding, newline=newline) as file_obj:
        writer = csv.writer(file_obj)
        if headers:
            writer.writerow(headers)
            for row in data:
                writer.writerow(row)
        else:
            writer.writerows(data)


def write_json(filepath, data, encoding="utf-8", ensure_ascii=False, indent=2):
    with open(filepath, "w", encoding=encoding) as file_obj:
        json.dump(data, file_obj, ensure_ascii=ensure_ascii, indent=indent)

### Get the data

__TODO__: Retrieve the player data from the CSV file and store it in a variable named `data`. Then
assign the first row to a variable named `headers` and the remaining rows to a variable named
`players`.

Note: `assert` statements are employed to verify the correctness of the data cleaning process. If
the assertions pass, the data cleaning process was successful. If any of the assertions fail, review
the cleaning/formatting functions.

The `# fmt: off ... # fmt: on` comments are used to disable/enable code formatters such as
[ruff](https://astral.sh/ruff) and [black](https://github.com/psf/black).

In [ ]:
filepath = "./data/data-2023-fifa_wwc-players.csv"
data = None  # Call function

headers = None  # Access headers nested list
players = None  # Access player nested lists

# fmt: off
assert headers == ["Rk", "Player", "Pos", "Squad", "Age", "Born", "90s", "Gls", "Sh", "SoT"]
assert players[-1] == ["619","Claudia Zornoza","MF","es Spain","32","1990","0.4","0","0","0"]
# fmt: on

## Skill practice: format/clean data

Three functions are provided to format and clean the player data:

`format_position()`: Reformats player's position string by converting the comma (`","`) delimiter
that separates multiple positions to a pipe (`|`), e.g., `"MF,DF"` -> `"MF|DF"`. This change
eliminates the need to surround the position string with double quotes when writing the value to a
CSV file.

`format_squad()`: Converts a player's "Squad" value (e.g. "es Spain") to a two-item tuple comprising
the following items: `(< Alpha-2 Code>, < Squad Name >)`. The
[Alpha-2](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-2) code is a two-character country code
that uniquely identifies the country associated with the squad.

`clean_player()`: "Cleans" a list representation of a player. Mutates targeted list elements by
converting numbers masquerading as strings to numeric types, and formatting position and squad
values. Delegates string formatting tasks to the function's `format_position()` and
`format_squad()`. The function assembles a new (local) list named `cleaned` with converted and
reformatted values. The function also extends the list with the two-item tuple returned from the
`format_squad()` call.

In [ ]:
def format_position(position):
    return position.replace(",", "|")


def format_squad(squad):
    return (squad[:2].upper(), squad[3:])


def clean_player(player, headers):
    cleaned = []
    for i, element in enumerate(player):
        if headers[i] in ("Rk", "Age", "Born", "Gls", "Sh", "SoT"):
            cleaned.append(int(element))
        elif headers[i] == "Pos":
            cleaned.append(format_position(element))
        elif headers[i] == "Squad":
            cleaned.extend(format_squad(element))  # Slice assignment
        elif headers[i] == "90s":
            cleaned.append(float(element))
        else:
            cleaned.append(element)

    return cleaned

### Clean the player lists

__TODO__: Cleaning each nested player list changes its length. This is the result of removing the
Alpha-2 country code (e.g., "CN") from the `Squad` value and adding it to the list as a separate
element. Consequently, a new string element "Alpha_2_Code" _must_ be inserted into the third (`3rd`)
position in the `headers` list to ensure that the `players` and `headers` indices remain
synchronized.

Call the appropriate list method to insert the string "Alpha_2_Code" into the `headers` list.

Add this line _after_ the `players` variable assignment.

If a runtime exception is encountered consider using the JupyterLab
[debugger](https://jupyterlab.readthedocs.io/en/stable/user/debugger.html) to
identify the issue.

In [ ]:
players = [clean_player(player, headers) for player in players]

# TODO: Insert "Alpha_2_Code" into the headers list


# fmt: off
assert headers == ["Rk", "Player", "Pos", "Alpha_2_Code", "Squad", "Age", "Born", "90s", "Gls", "Sh", "SoT"]
assert players[524] == [525, "Wang Shuang", "FW|MF", "CN", "China PR", 28, 1995, 1.8, 2, 3, 1]
# fmt: on

## Skill practice: access nested list elements

Three functions are provided to retrieve individual player data:

`get_player()`: Attempts to return a nested player list by employing the passed in player's name
(`"< last name > < first name>"`) as a filter. A case-insensitive string comparison is performed
when comparing names.

`get_player_position()`: Returns a player's position (i.e., "FW", "MF", "DF", "GK"). Note that
certain players are multi-position players. Their positions are separated by a pipe character
(`"|"`, e.g., `"MF|DF"`). The first position specified is considered the player's primary position.
The player's secondary position, if specified, can be returned by passing `primary=False`. If a
player lacks a secondary position, the player's primary position is always returned.

`get_player_shooting_stats()`: Returns a player's goals, shots, and shots on target. The target
elements in the `player` list are accesssed using a `slice` object.

__Check the docs__: If you have yet to utilize the built-in `slice` object, peruse the official
Python [documentation](https://docs.python.org/3/library/functions.html#slice) to understand its
purpose and usage.

In [ ]:
def get_player(players, name_idx, name):
    for player in players:
        if player[name_idx].lower() == name:
            return player


def get_player_position(player, pos_idx, primary=True):
    position = player[pos_idx]
    if "|" in position:
        primary_position, secondary_position = position.split("|")
        return primary_position if primary else secondary_position
    else:
        return position  # Single position


def get_player_shooting_stats(player, slice_):
    return [int(stat) for stat in player[slice_]]

### Kadidiatou Diani (France)

__TODO__: Utilize the `headers` list to look up the index of the "Players" header and assign the
value to `name_idx`. Then call the function `get_player()` and retrieve the nested list that
represents the French star Kadidiatou Diani. Assign the list to a variable named `diani`.

In [ ]:
name_idx = None  # Call object method
diani = None  # Call function

assert diani == [153, "Kadidiatou Diani", "FW", "FR", "France", 28, 1995, 5.0, 4, 17, 3]

__TODO__: Again, leverage the `headers` list to look up the index of the "Gls" (goals) header
element and assign the value to `gls_idx`. Access Diani's shooting statistics and unpack the return
value into the variables `diani_goals`, `diani_shots`, and `diani_shots_on_target`.

In [ ]:
gls_idx = None  # Call object method
diani_goals, diani_shots, diani_shots_on_target = None  # Call function

assert diani_goals == 4
assert diani_shots == 17
assert diani_shots_on_target == 3

### Marta Vieira da Silva (Brazil)

Brazil's all-time top goalscorer, Marta Vieira da Silva, also played in the 2023 FIFA Women's World
Cup (fifth World Cup appearance). The code below, accesses the nested list in `players` that
represents Marta (per the Brazilian tradition, filter on first name _only_). Assign the return
value to a variable named `marta`.

__TODO__: Take advantage of `headers` to look up the player position index. Assign the return value
to a variable named `pos_idx`.

The function `get_player_position()` is called __twice__ to access Marta's primary and secondary
positions and the return values are assigned to the variables `marta_primary_position` and
`marta_secondary_position`.

In [ ]:
marta = get_player(players, name_idx, "Marta Vieira da Silva")

pos_idx = None  # Call object method
marta_primary_position = get_player_position(marta, pos_idx)
martas_secondary_position = get_player_position(marta, pos_idx)

assert marta == [350, "Marta", "FW|MF", "BR", "Brazil", 37, 1986, 1.1, 0, 4, 3]
assert marta_primary_position == "FW"
assert martas_secondary_position == "MF"

### Defenders

__TODO__: Iterate over `players`. Inside the loop, include a conditional statement that evaluates
whether or not a player's __primary or secondary__ position is "DF" (defender). Utilize
`get_player_position()` when constructing the conditional statement.

If the conditional statement evaluates to `True`, add the player list to the list named `defenders`.

Employ a list comprehension to accomplish the task.

In [ ]:
defenders = []  # Populate list with defenders

# print(f"Defenders (primary or secondary position): {len(defenders)}")
# pp.pprint(defenders)

assert defenders == fxt.DEFENDERS

__Generative AI__: If the accompanying `assert` statement fails, utilize an AI chatbot to debug the
issue. Construct the prompt as follows:

1. Provide a brief description of the problem. Reference "Python" to help establish the context.
2. Include a slice of the `players` list. Ensure that at least one nested player list in the slice
   includes two positions (e.g., `"MF|DF"`). This will enrich the context and enhance the chatbot's
   understanding of the data.
3. Provide the offending code snippet.
4. Request assistance in debugging the issue.

### Write to file

__TODO__: Write the `defenders` list to a CSV file named "stu-2023-fifa_wwc-defenders.csv". Store
the file in the `data` subdirectory. Compare the file you produce to the associated CVS file named
"fxt-2023-fifa_wwc-defenders.csv". The two files must match, line for line, and character for
character.

In [ ]:
filepath = "./data/stu-2023-fifa_wwc-defenders.csv"
# TODO: Call function

## Skill practice: grouping data

The 2023 FIFA Women's World Cup featured thirty-two (`32`) squads. Each nested player list in
`players` includes both "Alpha_2_Code" and "Squad" elements. Locating all the player nested lists
in `players` that comprise a specific squad is handled by the function named `get_squad()`.

__TODO__: Implement the function. Read the description below to understand the function's expected
behavior.

`get_squad()`: Attempts to assemble a squad of nested player lists by employing the passed in
Alpha-2 country code as a filter. Parameters include:

* `players`: A list of nested player lists.
* `alpha_2_index`: "Alpha_2_Code" headers element index. Employed to access a
  player's Alpha-2 country code value.
* `alpha_2_code`: Alpha-2 country code employed as a filter.

A single line of code is all that is required to implement the function successfully.

In [ ]:
def get_squad(players, alpha_2_idx, alpha_2_code):
    pass  # TODO: Implement

### Tournament champion: Spain

__TODO__: Look up the "Alpha_2_Code" index in `headers`. Assign the return value of the method call
to the variable `alpha_2_idx`.

__TODO__: Call the function `get_squad()`, passing it the arguments required to retrieve the 2023
FIFA Women's World Cup champion: Spain's
[_La Roja_](https://en.wikipedia.org/wiki/Spain_women%27s_national_football_team)
squad. Assign the return value to a variable named `spain`.

In [ ]:
alpha_2_idx = None  # TODO call object method
spain = None  # TODO call function

print("Spain's La Roja squad")
pp.pprint(spain)

assert spain == fxt.SPAIN

## Goalscorers

How many players scored at least one (`1`) goal during the 2023 FIFA Women's World Cup? How many
players scored three or more (`3`+) goals during the tournament. Questions like these can be
answered by calling the function `get_goalscorers()`.

__TODO__: Implement the function. Read the description below to understand the function's expected
behavior.

`get_goalscorers()`: Returns a list of players who have scored a minimum `n` goals in the
tournament. Parameters include:

* `players`: A list of nested player lists.
* `gls_index`: "Gls" headers element index. Employed to access a player's
  goals value.
* `min_goals`: minimum goal threshold that a player _must_ meet to be included
  in the list. Default value is `1`.

A single line of code is all that is required to implement the function successfully.

In [ ]:
def get_goalscorers(players, gls_idx, min_goals=1):
    pass  # TODO: Implement

__TODO__: Exercise the function by returning a list of players who scored at least one goal (`1`+)
during the tournament. Assign the return value to the variable named `goalscorers.`

In [ ]:
goalscorers = None  # TODO: Call function
assert len(goalscorers) == 100

__TODO__: Call the function a second time. Pass it the arguments required to return a list of
players who scored at least three (`3`+) goals during the tournament. Assign the return value to the
variable named `goalscorers_3_plus`.

Sort `goalscorers_3_plus` __in-place__ per the following criteria and in the order specified:

1. goals scored (descending)
2. Shots on target (descending)
3. Shots (ascending)

Employ an anonymous `lambda` function to manage the sort.

In [ ]:
goalscorers_3_plus = None  # TODO: Call function

sh_idx = headers.index("Sh")
sot_idx = headers.index("SoT")

goalscorers_3_plus.sort()

print(f"Goalscorers (3+ goals): {len(goalscorers_3_plus)}")
pp.pprint(goalscorers_3_plus)

assert goalscorers_3_plus == fxt.GOALSCORERS_3_PLUS

__TODO__: Next, return a list of the English players who scored goals for the second-place (`2nd`)
[_Lionesses_](https://en.wikipedia.org/wiki/England_women%27s_national_football_team) squad. Assign
the return value of the function call to the variable named `england`. Then call `get_goalscorers()`
and return a list comprising all the English goalscorers. Assign the return value to the variable
named `england_goalscorers`.

Sort `england_goalscorers` _in-place_ per the sorting criteria applied earlier to
`goalscorers_3_plus`.

In [ ]:
england = None  # TODO: Call function
england_goalscorers = None  # TODO: Call function
england_goalscorers.sort()

pp.pprint(england_goalscorers)

assert england_goalscorers == fxt.ENGLAND_GOALSCORERS

## Skill practice: querying data

The FIFA Women's World Cup is a showcase for the world's top footballing talent. Several individual
awards are presented at the conclusion of the tournament, including the
[Golden Boot](https://en.wikipedia.org/wiki/FIFA_Women%27s_World_Cup_awards) for the top goalscorer.

The following three functions facilitate the identification of the tournament's top goalscorer(s) as
well as leading goalscorer(s) for each squad:

`get_country_codes()`: Returns a list of Alpha-2 country codes associated with participating squads
(n=`33`) sourced from the nested player lists in `players`. Code uniqueness is guaranteed by
converting the list to a set. Codes are sorted in ascending order before being returned to the
caller.

`get_top_goalscorer()`: Returns the top tournament goalscorer(s) in the passed in `players` list.
The possibility exists that one or more players shares top scorer honors. In such cases, the
`top_scorers` dictionary is provisioned with a "players" list for referencing more than one top
goalscorer. When the loop terminates the `top_scorers` dictionary is returned to the caller.

`get_top_goalscorers_by_country()`: Leverages the Alpha-2 country codes to return the top
goalscorer(s) for each squad/country. Delegates to the functions `get_country_codes()` and
`get_top_goalscorer()` the tasks of retrieving the country codes to iterate over and returning each
squad's top goalscorer(s). The function utilizes the `**` unpacking operator to unpack the
"top scorer" dictionary returned by calling `get_top_goalscorer()` into each new "country"
dictionary included in the list returned to the caller.

In [ ]:
def get_country_codes(players, alpha_2_idx):
    return sorted(set(player[alpha_2_idx] for player in players))


def get_top_goalscorer(players, gls_idx):
    top_scorer = {"top_scorer_goals": 0, "top_scorers": []}
    for player in players:
        goals = int(player[gls_idx])
        if goals > top_scorer["top_scorer_goals"]:
            top_scorer["top_scorer_goals"] = goals
            top_scorer["top_scorers"] = [player]
        elif goals == top_scorer["top_scorer_goals"]:
            top_scorer["top_scorers"].append(player)
    return top_scorer["top_scorers"]


def get_top_goalscorers_by_country(players, alpha_2_idx, gls_idx):
    return [
        {"country": code, **get_top_goalscorer(get_squad(players, alpha_2_idx, code), gls_idx)}
        for code in get_country_codes(players, alpha_2_idx)
    ]

### Golden Boot Award

__TODO__: Identify the player or players&mdash;ties are possible&mdash;awarded the
[Golden Boot](https://en.wikipedia.org/wiki/FIFA_Women%27s_World_Cup_awards) for scoring the most
goals during the 2023 FIFA Women's World Cup.

Note: A counterfactural scenario is presented in the second `assert` statement. Our very own
Elle O'Brien joins the US squad and delivers a standout tournament performance. Such is her scoring
prowess that she matches the tournament's top goalscorer. Both players share the Golden Boot Award.
Our "shared" assertion is designed to test the function's ability to accommodate ties.

In [ ]:
top_scorer = get_top_goalscorer(players, gls_idx)  # Hinata Miyazawa (Japan)

print(f"Top Scorer: (n={len(top_scorer['top_scorers'])})")
pp.pprint(top_scorer)

assert top_scorer == fxt.GOLDEN_BOOT
# Test a counterfactual shared award
fxt_players = fxt.GOALSCORERS_3_PLUS + [fxt.ELLE]  # Add Elle O'Brien
assert get_top_goalscorer(fxt_players, gls_idx) == fxt.GOLDEN_BOOT_SHARED

### Alpha-2 country codes

__TODO__: Next, test the function `get_country_codes()` by passing it the arguments required to
return a list of unique Alpha-2 country codes. Assign the return value to the variable named
`country_codes`.

In [ ]:
country_codes = get_country_codes(players, alpha_2_idx)

# print(f"Alpha-2 Country Codes (n={len(country_codes)})")
# pp.pprint(country_codes)

assert len(country_codes) == 32
assert country_codes == fxt.COUNTRY_CODES

### Top goalscorers by squad

__TODO__: Return the top goalscorer(s) for each participating squad. Assign the return value to the
variable named `top_scorers`.

In [ ]:
top_scorers = None  # Call function

print(f"\nTop scorers by Country (n={len(top_scorers)})")
pp.pprint(top_scorers)

assert len(top_scorers) == 30
assert top_scorers == fxt.COUNTRY_TOP_SCORERS

### Write to file

__TODO__: Write `top_scorers` to a JSON file named "stu-2023-fifa_wwc-top_scorers.json". Store the
file in the `data` subdirectory. Compare the file you produce to the associated JSON file named
"fxt-2023-fifa_wwc-top_scorers.json". The two files must match, line for line, and character for
character.

In [ ]:
filepath = "./data/stu-2023-fifa_wwc-top_scorers.json"
# TODO: Call function

## Skill practice: calculating conversion rates

One metric of player effectiveness is the shot conversion rate. The shot conversion rate is calculated by dividing the number of goals (`Gls`) by the total number of shots (`Sh`) taken by a player, providing insight into how effectively a player converts their attempts into goals. A related metric, the shots on target (`SoT`) conversion rate, focuses on the quality of these attempts by considering only the shots that would have resulted in a goal if not blocked by the goalkeeper or another player considered the "last defender" (shots that strike the goal post or crossbar are not considered a shot on target).

The following function calculates the conversion rate for either type of shot:

`get_shot_conversion_rate()`: Calculates the conversion rate for player shots (`Sh`) and shots on target (`SoT`). Either number can be passed as the `shots` argument. The number of decimal places to retain when rounding the quotient is specified by the `precision` argument.

In [ ]:
def get_shot_conversion_rate(goals, shots, precision=2):
    return round(goals / shots, precision)

### Hinata Miyazaki (Japan)

The code below retrieves the Japanese midfielder Hinata Miyazaki, computes her shots and shots on
target conversion rates, rounding the results to three (`3`) decimal places.

In [ ]:
# Get player
miyazawa = get_player(players, name_idx, "Hinata Miyazawa")

# Get goals, shots, and shots on target
miyazawa_goals, miyazawa_shots, miyazawa_shots_on_target = get_player_shooting_stats(
    miyazawa, slice(gls_idx, len(headers))
)

# Compute shot conversion rate
miyazawa_shot_conv_rate = get_shot_conversion_rate(miyazawa_goals, miyazawa_shots, precision=3)

# Compute shots on target conversion rate
miyazawa_shot_target_conv_rate = get_shot_conversion_rate(
    miyazawa_goals, miyazawa_shots_on_target, precision=3
)

assert miyazawa == [374, "Hinata Miyazawa", "FW|MF", "JP", "Japan", 23, 1999, 3.7, 5, 13, 6]
assert miyazawa_goals == 5
assert miyazawa_shots == 13
assert miyazawa_shots_on_target == 6
assert miyazawa_shot_conv_rate == 0.385
assert miyazawa_shot_target_conv_rate == 0.833

### Naomi Girma (United States)

The code below confirms that the function can handle players who notched zero `0` goals during the
tournament by computing US defender Naomi Girma's shot and shots on target conversion rates.

Note: Naomi is considered one of the world's top defenders. Stopping opponents from scoring is her
primary objective.

In [ ]:
# Get player
girma = get_player(players, name_idx, "Naomi Girma")

# Shooting stats
girma_goals, girma_shots, girma_shots_on_target = get_player_shooting_stats(
    girma, slice(gls_idx, len(headers))
)

# Shot conversion rate
girma_shot_conv_rate = get_shot_conversion_rate(girma_goals, girma_shots)

# Shots on target conversion rate
girma_shot_target_conv_rate = get_shot_conversion_rate(girma_goals, girma_shots_on_target)

assert girma == [212, "Naomi Girma", "DF", "US", "USA", 23, 2000, 4.3, 0, 0, 0]
assert girma_goals == 0
assert girma_shots == 0
assert girma_shots_on_target == 0
assert girma_shot_conv_rate == 0.0
assert girma_shot_target_conv_rate == 0.0

If any of the `assert` statements fail, review each of the functions employed
in the cell. Ensure that the correct arguments are passed to each function and
that the expected return values are assigned to the appropriate variables.

### All goalscorers

Finally, the code below computes the shot and shots on target conversion rates for all tournament
goalscorers, rounding each result to two (`2`) decimal places. The list is then sorted __in-place__
per the following criteria and in the order specified:

1. goals scored (descending)
2. shots on target conversion rate (descending)
3. shot conversion rate (ascending)
4. player name (ascending)

__TODO__: Employ an anonymous `lambda` function to manage the sort.

__TODO__: Remember to also extend the `headers` with the following header names: "Sh_Conv_Rate", and
"SoT_Conv_Rate".

In [ ]:
for player in goalscorers:
    goals, shots, shots_on_target = get_player_shooting_stats(player, slice(gls_idx, len(headers)))
    shot_conv_rate = get_shot_conversion_rate(goals, shots)
    shots_on_target_conv_rate = get_shot_conversion_rate(goals, shots)
    player.extend([shot_conv_rate, shots_on_target_conv_rate])

goalscorers.sort()

# TODO: Extend the headers

print(goalscorers[:5])

In [ ]:
# fmt: off
assert headers == [
    "Rk", "Player", "Pos", "Alpha_2_Code", "Squad", "Age", "Born",
    "90s", "Gls", "Sh", "SoT", "Sh_Conv_Rate", "SoT_Conv_Rate",
]

assert goalscorers[:5] == [
    [374, "Hinata Miyazawa", "FW|MF", "JP", "Japan", 23, 1999, 3.7, 5, 13, 6, 0.38, 0.83],
    [447, "Alexandra Popp", "FW|MF", "DE", "Germany", 32, 1991, 2.9, 4, 14, 7, 0.29, 0.57],
    [485, "Jill Roord", "MF", "NL", "Netherlands", 26, 1997, 4.6, 4, 18, 7, 0.22, 0.57],
    [270, "Amanda Ilestedt", "DF", "SE", "Sweden", 30, 1993, 7.0, 4, 9, 6, 0.44, 0.67],
    [153, "Kadidiatou Diani", "FW", "FR", "France", 28, 1995, 5.0, 4, 17, 3, 0.24, 1.33],
]
# fmt: on

### Write to file

__TODO__: Write the mutated `goalscorers` list to a CSV file named
"stu-2023-fifa_wwc-goalscorers.csv". Store the file in the `data` subdirectory. Compare the file you
produce to the associated CVS file named "fxt-2023-fifa_wwc-goalscorers.csv". The two files must
match, line for line, and character for character.

In [ ]:
filepath = "./data/stu-2023-fifa_wwc-goalscorers.csv"
# TODO: Call function